In [18]:
from pathlib import Path
import sys

# Add parent directory to Python path
parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [19]:
import geopandas as gpd 
import pandas as pd
import os 

import routing 
import osm 

import osmnx as ox
import co2

In [20]:
aoi = gpd.read_file("../aoi/aoi.gpkg")
aoi = aoi.to_crs(aoi.estimate_utm_crs())
aoi

,Name,type,population,osmid,distance_to_node,geometry
0,Frankfurt Bezirk 27,Stadtteil,8908.140428,31941674,121.687367,"MULTIPOLYGON (((478966.135 5552679.23, 478964...."
1,Frankfurt Bezirk 28,Stadtteil,6632.463686,31710439,33.489584,"MULTIPOLYGON (((479099.926 5552327.127, 479084..."
2,Bockenheim,Stadtteil,32196.788691,11148720833,11.952396,"MULTIPOLYGON (((474137.825 5551457.352, 474134..."
3,Bonames,Stadtteil,12635.036304,27405627,46.314649,"MULTIPOLYGON (((475759.938 5557724.818, 475757..."
4,Frankfurt Bezirk 31,Stadtteil,1066.817343,415585886,68.327220,"MULTIPOLYGON (((477652.623 5550375.63, 477650...."
...,...,...,...,...,...,...
94,Odenwaldkreis,Landkreis,96191.774804,269421067,1308.142115,"MULTIPOLYGON (((488539.291 5500844.059, 488539..."
95,Fulda,Landkreis,227488.768518,1657720727,1397.636176,"MULTIPOLYGON (((550109.983 5578919.207, 550063..."
96,Vogelsberg,Landkreis,102957.513896,352980044,6746.543470,"MULTIPOLYGON (((507642.328 5586973.323, 507488..."
97,Rheinland-Pfalz,Bundesland,934385.526148,311638106,3289.258385,"MULTIPOLYGON (((489148.582 5483259.864, 489030..."


In [21]:
aoi_city = aoi[aoi["Name"].str.contains("Frankfurt", na=False)].reset_index(drop=True)

In [22]:
streets_graph_path = os.path.normpath("../streets/streets.graphml")
osm_xml_file = os.path.normpath("../streets/streets.osm")
results_path = "../streets"

In [23]:
os.makedirs(results_path,exist_ok=True)

In [24]:
# Run only if you have osmium installed

# Select what type of street network you want to load
network_filter = osm.osmium_network_filter("drive")
# Download the region pbf file crop it by aoi and convert to osm format
osm.geofabrik_to_osm(
    osm_xml_file,
    input_file=results_path,
    aoi=aoi,
    osmium_filter_args=network_filter,
    overwrite=False
)

File '../streets/streets.osm' already exists. Skipping conversion.


'../streets/streets.osm'

In [47]:
if os.path.isfile(streets_graph_path):
    G = ox.load_graphml(streets_graph_path)
else:
    # Load
    G = ox.graph_from_xml(osm_xml_file)
    # Project geometry coordinates to UTM system to allow euclidean meassurements in meters (sorry americans)
    G = ox.project_graph(G,to_crs=aoi.estimate_utm_crs())
    # Save the graph in graphml format to avoid the slow loading process
    ox.save_graphml(G,streets_graph_path)

In [48]:
nodes,edges = ox.graph_to_gdfs(G)

In [49]:
node_penalty = 5 
acceleration = 1.5
min_cruising_time = 5
min_cruising_speed = 10
max_stop_and_go_speed = 50

maxspeeds = {
    "living_street": 30,
    "motorway": 100,
    "motorway_link": 60,
    "primary": 50,
    "primary_link": 50,
    "residential": 30,
    "secondary": 40,
    "secondary_link": 40,
    "service": 20,
    "tertiary": 40,
    "tertiary_link": 40,
    "trunk": 80,
    "trunk_link": 60,
    "unclassified": 40
}

In [50]:
edges["maxspeed_car"] = routing.infer_maxspeed(edges,maxspeeds,enforce=False)
edges["travel_time_car"], edges["avg_speed_car"] = routing.travel_time(
    edges=edges,
    acceleration=acceleration,
    min_cruising_speed=min_cruising_speed,
    min_cruising_time=min_cruising_time,
    max_stop_and_go_speed=max_stop_and_go_speed,
    node_penalty=node_penalty,
    maxspeed_col="maxspeed_car",
    return_speed=True
)
edges["co2_car"] = co2.route_hbefa(
    edges,
    avg_speed_col="avg_speed_car",
    vehicle_type="gasoline_pc",
    maxspeed_col="maxspeed_car",
    return_total=False
)

In [51]:
ROUTE_TYPE_PRIORITY = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "service",
    "living_street",
    "residential",
    "unclassified",
]

def normalize_route_type(route_type):
    if isinstance(route_type, (list, tuple)):
        return next(
            (rt for rt in ROUTE_TYPE_PRIORITY if rt in route_type),
            None
        )
    return route_type  # already a single value

edges["highway"] = edges["highway"].map(normalize_route_type)

In [52]:
# ----------------------------
# 1. Filter edges
# ----------------------------

highway_filter = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link"
]
highway_filter_city = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
]

edges_filtered = edges[
    edges.intersects(aoi_city.union_all()) |
    (
        edges.intersects(aoi_city.union_all().buffer(10000)) & 
        edges["highway"].isin(highway_filter_city)
    ) |
    edges["highway"].isin(highway_filter)
]
u_set = set(edges_filtered.index.get_level_values("u"))
v_set = set(edges_filtered.index.get_level_values("v"))
mask = (
    (edges.index.get_level_values("u").isin(u_set) |
    edges.index.get_level_values("v").isin(v_set) |
    edges.index.get_level_values("u").isin(v_set) |
    edges.index.get_level_values("v").isin(u_set)) &
    ~edges.index.isin(edges_filtered.index)
)

oneway_edges = edges[mask] 
oneway_edges["oneway"] = False
oneway_edges_swapped = oneway_edges.copy()
idx = oneway_edges_swapped.index
oneway_edges_swapped.index = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values('v'),
        idx.get_level_values('u'),
        idx.get_level_values('key')
    ],
    names=['u', 'v', 'key']
)

edges_filtered = pd.concat([oneway_edges, oneway_edges_swapped, edges_filtered])
edges_filtered = edges_filtered[~edges_filtered.index.duplicated(keep='first')]

In [53]:
from sklearn.cluster import DBSCAN
import numpy as np

border_nodes = nodes[~nodes.intersects(aoi.union_all())]
idx = edges.index

reversed_pairs = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values("v"),
        idx.get_level_values("u"),
    ],
    names=["u", "v"]
)

existing_pairs = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values("u"),
        idx.get_level_values("v"),
    ],
    names=["u", "v"]
)

mask = (
    (
        idx.get_level_values("u").isin(border_nodes.index) |
        idx.get_level_values("v").isin(border_nodes.index)
    ) &
    (
        ~reversed_pairs.isin(existing_pairs)
    )
)
border_edges = edges[mask]

border_nodes = border_nodes[
    border_nodes.index.isin(border_edges.index.get_level_values("u")) |
    border_nodes.index.isin(border_edges.index.get_level_values("v"))
]


coords = np.array([
    (geom.x, geom.y)
    for geom in border_nodes.geometry
])

border_nodes["cluster_id"] = DBSCAN(
    eps=5000,
    min_samples=1
).fit_predict(coords)

border_nodes["cluster_id"] = (
    border_nodes.groupby("cluster_id")["cluster_id"]
    .transform(lambda s: s.index[0])
)

connecting_edges_u = border_edges.copy()
idx = connecting_edges_u.index
u_mapping = border_nodes["cluster_id"].to_dict()
connecting_edges_u.index = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values("u").map(u_mapping),
        idx.get_level_values("v"),
        idx.get_level_values("key"),
    ],
    names=["u", "v", "key"]
)
connecting_edges_u = connecting_edges_u[
    connecting_edges_u.index.get_level_values("u").notna()
]

connecting_edges_v = border_edges.copy()
idx = connecting_edges_v.index
v_mapping = border_nodes["cluster_id"].to_dict()
connecting_edges_v.index = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values("u"),
        idx.get_level_values("v").map(v_mapping),
        idx.get_level_values("key"),
    ],
    names=["u", "v", "key"]
)
connecting_edges_v = connecting_edges_v[
    connecting_edges_v.index.get_level_values("v").notna()
]

edges_filtered = pd.concat([
    edges_filtered,
    connecting_edges_u,
    connecting_edges_v
])
edges_filtered = edges_filtered[~edges_filtered.index.duplicated(keep='first')]

In [61]:
connected_nodes = set(edges_filtered.index.get_level_values("u")).union(set(edges_filtered.index.get_level_values("v")))
nodes_filtered = nodes[nodes.index.isin(connected_nodes)]
nodes_filtered.index = nodes_filtered.index.astype(int)
idx = edges_filtered.index

edges_filtered.index = pd.MultiIndex.from_arrays(
    [
        idx.get_level_values("u").astype(int),
        idx.get_level_values("v").astype(int),
        idx.get_level_values("key").astype(int),
    ],
    names=idx.names
)
H = ox.graph_from_gdfs(nodes_filtered,edges_filtered) 
H = ox.truncate.largest_component(H,strongly=True)
ox.save_graphml(H,streets_graph_path)
nodes, edges = ox.graph_to_gdfs(H)

In [62]:
edges.to_file("edges.gpkg")

In [63]:
aoi_copy = aoi.copy()
aoi_copy.geometry = aoi_copy.geometry.centroid
aoi["osmid"] = routing.nearest_nodes(aoi_copy,H)
node_geom = nodes.geometry.loc[aoi["osmid"]].values

aoi["distance_to_node"] = aoi.geometry.centroid.distance(
    gpd.GeoSeries(node_geom, index=aoi.index, crs=aoi.crs)
)
aoi.to_file("../aoi/aoi.gpkg")